# 38 — Comparación conjunta de Burgers, onda y Allen–Cahn

**Implementado, sin ejecutar.** Este notebook reúne los resultados de los notebooks 35–37. No contiene llamadas a entrenamiento. Exige 150 corridas completas, de cinco métodos y diez semillas en cada problema. Si faltan resultados falla explícitamente; nunca intenta producirlos mediante entrenamiento.

In [ ]:
from pathlib import Path
import sys, os, json, subprocess
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['PYTHONIOENCODING'] = 'utf-8'
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
config = json.loads((ROOT / 'pca_collocation/configs/benchmark_full.json').read_text(encoding='utf-8'))
OUT = ROOT / 'pca_collocation/results/benchmark' / config['run_id']
COMMAND = [sys.executable, '-u', str(ROOT / 'scripts/run_collocation_benchmark.py')]
print('Python:', sys.executable)
print('Resultados:', OUT)
print('Semillas:', config['seeds'])


## Auditar y reunir

Se verifica finalización de cada corrida, protocolo congelado, dimensiones, puntos únicos, restricciones duras, selección del mejor modelo por residuo de validación y selecciones QR reconstruidas sobre historias tempranas y tardías. Se conservan todas las semillas. No se mezclan resultados con el piloto ni con estudios anteriores.

In [ ]:
subprocess.run(COMMAND + ['--aggregate'], cwd=ROOT, check=True)


## Comparación entre problemas

Se presentan medias, DE, medianas y distribuciones individuales por problema, junto con diferencias pareadas PCA + media menos cada control. Se conserva el terminal como contraste secundario. No se calcula un promedio global de RMSE o residuos de PDE distintas. Las posiciones de cada método por problema son descriptivas; los intervalos al 95 % no están ajustados por las doce comparaciones. Una posición promedio favorable no demuestra superioridad universal.

In [ ]:
from IPython.display import Image, display
import pandas as pd
RESULTS = OUT / 'aggregate'
assert (RESULTS / 'report_complete.json').exists(), 'Falta completar la agregación'
summary = pd.read_csv(RESULTS / 'summary.csv')
display(summary[summary['checkpoint'] == 'best_validation'])
display(pd.read_csv(RESULTS / 'ranks_by_problem.csv'))
print((RESULTS / 'paired_comparisons.json').read_text(encoding='utf-8'))
for name in ('error_comparison.png', 'validation_curves.png'):
    display(Image(filename=str(RESULTS / name), format='png', embed=True))


## Archivos reproducibles

Ejecutar desde terminal: `python scripts/run_collocation_benchmark.py --aggregate`. Se escribe el subdirectorio aggregate/: metrics.csv, summary.csv, paired_individual.csv, paired_comparisons.json, ranks_by_problem.csv, traces.csv, pca_rank_diagnostics.csv, audit.json y figuras PNG. Se genera también 38_benchmark_aggregate_results.ipynb con figuras incrustadas. Repetir el comando solo recalcula auditorías y reportes. Los archivos originales por corrida se conservan. El comando se ejecuta después de los tres problemas; no inicia redes si faltan datos.